# Sistema Avançado de Detecção de Fadiga - Versão Inteligente

## Características Avançadas
- **Perfis Personalizados**: Cada usuário tem seu próprio perfil calibrado
- **Histórico Inteligente**: Aprende com sessões anteriores
- **Regras Adaptativas**: Ajusta-se ao comportamento individual
- **Análise Temporal**: Detecta padrões complexos ao longo do tempo
- **Sistema de Alertas**: Progressivo com múltiplos níveis
- **Métricas Personalizadas**: Thresholds únicos por pessoa

## 🔬 Funcionamento Inteligente
1. **Calibração Inicial**: 30s para aprender padrões pessoais
2. **Detecção Adaptativa**: Regras se ajustam ao usuário
3. **Histórico Persistente**: Salva dados entre sessões
4. **Alertas Progressivos**: Aviso → Atenção → Alerta → Crítico
5. **Análise Comportamental**: Detecta mudanças sutis

In [10]:
# 📦 Instalação de dependências para alerta sonoro
# Execute esta célula caso pygame não esteja instalado
import subprocess
import sys

def instalar_pygame():
    try:
        import pygame
        print("✅ pygame já está instalado")
        pygame.mixer.init()
        print("✅ pygame.mixer funcionando corretamente")
        return True
    except ImportError:
        print("📦 Instalando pygame...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "pygame"])
            print("✅ pygame instalado com sucesso!")
            return True
        except Exception as e:
            print(f"❌ Erro ao instalar pygame: {e}")
            print("💡 O sistema usará beep do sistema como fallback")
            return False
    except Exception as e:
        print(f"⚠️ pygame instalado mas com problemas: {e}")
        print("💡 O sistema usará beep do sistema como fallback")
        return False

def verificar_dependencias():
    print("🔍 Verificando dependências do alerta sonoro...")
    
# Verificar pygame
    pygame_ok = instalar_pygame()
    
# Verificar numpy (já deve estar instalado)
    try:
        import numpy
        print("✅ numpy disponível")
        numpy_ok = True
    except ImportError:
        print("❌ numpy não encontrado (necessário para síntese de som)")
        numpy_ok = False
    
# Verificar threading (built-in)
    try:
        import threading
        print("✅ threading disponível")
        threading_ok = True
    except ImportError:
        print("❌ threading não encontrado")
        threading_ok = False
    
    print("📊 Resumo das dependências:")
    print(f"  pygame: {'✅' if pygame_ok else '❌'}")
    print(f"  numpy: {'✅' if numpy_ok else '❌'}")
    print(f"  threading: {'✅' if threading_ok else '❌'}")
    
    if pygame_ok and numpy_ok:
        print("🎵 Sistema de alerta sonoro totalmente funcional!")
    elif threading_ok:
        print("🔊 Sistema usará beep do sistema (funcional mas limitado)")
    else:
        print("⚠️ Alertas sonoros podem não funcionar corretamente")
    
    return pygame_ok and numpy_ok

# Executar verificação
if __name__ == "__main__":
    verificar_dependencias()

🔍 Verificando dependências do alerta sonoro...
✅ pygame já está instalado
✅ pygame.mixer funcionando corretamente
✅ numpy disponível
✅ threading disponível
📊 Resumo das dependências:
  pygame: ✅
  numpy: ✅
  threading: ✅
🎵 Sistema de alerta sonoro totalmente funcional!


In [11]:
# Imports e configurações
import cv2
import numpy as np
import mediapipe as mp
import time
import json
import os
from collections import deque, defaultdict
from scipy.spatial.distance import euclidean
from scipy import stats
import joblib
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import threading
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliotecas carregadas")
print("🧠 Sistema Inteligente Inicializado")

✅ Bibliotecas carregadas
🧠 Sistema Inteligente Inicializado


In [16]:
# Classe para gerenciar perfis de usuários - expandida
class GerenciadorPerfis:
    def __init__(self, diretorio_perfis="perfis_usuarios"):
        self.diretorio = Path(diretorio_perfis)
        self.diretorio.mkdir(exist_ok=True)
        self.perfil_atual = None
    
    def converter_numpy_para_python(self, obj):
        """Converte tipos NumPy para tipos Python nativos (JSON serializável)"""
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, dict):
            return {key: self.converter_numpy_para_python(value) for key, value in obj.items()}
        elif isinstance(obj, list):
            return [self.converter_numpy_para_python(item) for item in obj]
        else:
            return obj
    
    def criar_perfil(self, nome_usuario):
        """Cria um novo perfil de usuário expandido"""
        perfil = {
            'nome': nome_usuario,
            'criado_em': datetime.now().isoformat(),
            'sessoes': [],
            'calibracao': {
                'ear_baseline': None,
                'ear_std': None,
                'mar_baseline': None,
                'mar_std': None,
                'blink_rate_baseline': None,
                'head_stability_baseline': None,  # ⚡ NOVA MÉTRICA ⚡
                'perclos_alerta_max': 20.0,
                'perclos_sonolento_min': 60.0,
                'calibrado': False
            },
# Thresholds com novas métricas
            'thresholds_personalizados': {
# Thresholds tradicionais
                'ear_sonolento': 0.25,
                'ear_critico': 0.20,
                'perclos_aviso': 30.0,
                'perclos_atencao': 45.0,
                'perclos_alerta': 60.0,
                'perclos_critico': 80.0,
                'tempo_olhos_fechados_aviso': 1.5,
                'tempo_olhos_fechados_critico': 3.0,
                
# novos thresholds avançados
# Mar e detecção de bocejos
                'mar_bocejo_moderado': 0.30,
                'mar_bocejo_extremo': 0.40,
                'duracao_bocejo_minima': 1.0,
                'duracao_bocejo_critica': 3.0,
                
# Blink_Rate
                'blink_rate_muito_baixa': 3.0,
                'blink_rate_baixa': 5.0,
                'blink_rate_alta': 25.0,
                'blink_rate_muito_alta': 30.0,
                'blink_rate_normal_min': 8.0,
                'blink_rate_normal_max': 22.0,
                
# Head_Stability
                'head_stability_leve': 7.0,
                'head_stability_moderada': 10.0,
                'head_stability_alta': 15.0,
                
# Combinações e eventos especiais
                'microsleep_perclos_threshold': 80.0,
                'microsleep_blink_rate_threshold': 2.0,
                'fadiga_severa_indicators_required': 3,
                'eventos_simultaneos_threshold': 2,
                
# Contadores de eventos
                'max_bocejos_sessao': 2,
                'max_bocejos_critico': 3
            },
# estatísticas expandidas
            'estatisticas': {
                'total_sessoes': 0,
                'tempo_total_uso': 0,
                'alertas_gerados': 0,
                'ear_medio_historico': [],
                'perclos_medio_historico': [],
# Novas estatísticas
                'blink_rate_medio_historico': [],
                'head_stability_medio_historico': [],
                'total_bocejos_historico': [],
                'total_blinks_historico': [],
                'scores_fadiga_historico': []
            }
        }
        
        arquivo_perfil = self.diretorio / f"{nome_usuario}.json"
        with open(arquivo_perfil, 'w') as f:
            json.dump(perfil, f, indent=2)
        
        self.perfil_atual = perfil
        print(f"✅ Perfil expandido criado para {nome_usuario}")
        return perfil
    
    def carregar_perfil(self, nome_usuario):
        """Carrega perfil existente e atualiza com novos campos se necessário"""
        arquivo_perfil = self.diretorio / f"{nome_usuario}.json"
        if arquivo_perfil.exists():
            with open(arquivo_perfil, 'r') as f:
                self.perfil_atual = json.load(f)
            
# migração automática: Adicionar novos campos se não existirem
            self.migrar_perfil_para_versao_avancada()
            
            print(f"✅ Perfil carregado para {nome_usuario}")
            print(f"   📊 {len(self.perfil_atual['sessoes'])} sessões anteriores")
            print(f"   🎯 Calibrado: {'Sim' if self.perfil_atual['calibracao']['calibrado'] else 'Não'}")
            return self.perfil_atual
        else:
            print(f"⚠️ Perfil {nome_usuario} não existe, criando novo...")
            return self.criar_perfil(nome_usuario)
    
    def migrar_perfil_para_versao_avancada(self):
        """NOVA FUNÇÃO: Migra perfis antigos para incluir novos campos """
        if not self.perfil_atual:
            return
        
        migrou = False
        
# adicionar novos campos se necessário
        calibracao = self.perfil_atual.get('calibracao', {})
        if 'head_stability_baseline' not in calibracao:
            calibracao['head_stability_baseline'] = None
            migrou = True
        
# adicionar novos thresholds
        thresholds = self.perfil_atual.get('thresholds_personalizados', {})
        novos_thresholds = {
            'mar_bocejo_moderado': 0.30,
            'mar_bocejo_extremo': 0.40,
            'duracao_bocejo_minima': 1.0,
            'duracao_bocejo_critica': 3.0,
            'blink_rate_muito_baixa': 3.0,
            'blink_rate_baixa': 5.0,
            'blink_rate_alta': 25.0,
            'blink_rate_muito_alta': 30.0,
            'blink_rate_normal_min': 8.0,
            'blink_rate_normal_max': 22.0,
            'head_stability_leve': 7.0,
            'head_stability_moderada': 10.0,
            'head_stability_alta': 15.0,
            'microsleep_perclos_threshold': 80.0,
            'microsleep_blink_rate_threshold': 2.0,
            'fadiga_severa_indicators_required': 3,
            'eventos_simultaneos_threshold': 2,
            'max_bocejos_sessao': 2,
            'max_bocejos_critico': 3
        }
        
        for chave, valor in novos_thresholds.items():
            if chave not in thresholds:
                thresholds[chave] = valor
                migrou = True
        
# Adicionar novas estatísticas se não existirem
        estatisticas = self.perfil_atual.get('estatisticas', {})
        novas_estatisticas = {
            'blink_rate_medio_historico': [],
            'head_stability_medio_historico': [],
            'total_bocejos_historico': [],
            'total_blinks_historico': [],
            'scores_fadiga_historico': []
        }
        
        for chave, valor in novas_estatisticas.items():
            if chave not in estatisticas:
                estatisticas[chave] = valor
                migrou = True
        
        if migrou:
            self.salvar_perfil()
            print("🔄 Perfil migrado para versão avançada")
    
    def salvar_perfil(self):
        """Salva perfil atual com conversão JSON-safe"""
        if self.perfil_atual:
            perfil_convertido = self.converter_numpy_para_python(self.perfil_atual)
            
            arquivo_perfil = self.diretorio / f"{self.perfil_atual['nome']}.json"
            with open(arquivo_perfil, 'w') as f:
                json.dump(perfil_convertido, f, indent=2)
    
    def atualizar_calibracao(self, ear_dados, mar_dados, blink_dados, head_stability_dados=None):
        """CALIBRAÇÃO EXPANDIDA: Atualiza calibração com todas as métricas """
        if not self.perfil_atual:
            return
        
        calibracao = self.perfil_atual['calibracao']
        calibracao['ear_baseline'] = float(np.mean(ear_dados))
        calibracao['ear_std'] = float(np.std(ear_dados))
        calibracao['mar_baseline'] = float(np.mean(mar_dados))
        calibracao['mar_std'] = float(np.std(mar_dados))
        calibracao['blink_rate_baseline'] = float(np.mean(blink_dados))
        
# HEAD STABILITY
        if head_stability_dados and len(head_stability_dados) > 0:
            calibracao['head_stability_baseline'] = float(np.mean(head_stability_dados))
        else:
            calibracao['head_stability_baseline'] = 5.0  # Valor padrão
        
        calibracao['calibrado'] = True
        
# ajustar thresholds baseados na calibração
        thresholds = self.perfil_atual['thresholds_personalizados']
        ear_baseline = calibracao['ear_baseline']
        ear_std = calibracao['ear_std']
        blink_baseline = calibracao['blink_rate_baseline']
        head_stability_baseline = calibracao['head_stability_baseline']
        
# Thresholds adaptativos baseados nos dados pessoais - ConversÃO Float
        thresholds['ear_sonolento'] = float(max(0.20, ear_baseline - 2 * ear_std))
        thresholds['ear_critico'] = float(max(0.15, ear_baseline - 3 * ear_std))
        
# novos thresholds adaptativos
# Blink rate personalizado (±30% do baseline individual)
        thresholds['blink_rate_baixa'] = float(max(3.0, blink_baseline * 0.4))
        thresholds['blink_rate_muito_baixa'] = float(max(2.0, blink_baseline * 0.2))
        thresholds['blink_rate_alta'] = float(min(30.0, blink_baseline * 1.6))
        thresholds['blink_rate_muito_alta'] = float(min(40.0, blink_baseline * 2.0))
        
# Head stability personalizado (baseline + desvios)
        thresholds['head_stability_leve'] = float(head_stability_baseline + 2.0)
        thresholds['head_stability_moderada'] = float(head_stability_baseline + 5.0)
        thresholds['head_stability_alta'] = float(head_stability_baseline + 10.0)
        
        print(f"🎯 Calibração avançada atualizada:")
        print(f"   👁️ EAR baseline: {ear_baseline:.3f} ± {ear_std:.3f}")
        print(f"   📊 MAR baseline: {calibracao['mar_baseline']:.3f}")
        print(f"   👁️ Blink rate baseline: {blink_baseline:.1f}/min")
        print(f"   🎯 Head stability baseline: {head_stability_baseline:.2f}")
        print(f"   🎯 Thresholds personalizados definidos")
        
        self.salvar_perfil()
    
    def adicionar_sessao(self, dados_sessao):
        """SESSÃO EXPANDIDA: Adiciona dados com novas métricas """
        if not self.perfil_atual:
            return
        
        dados_convertidos = self.converter_numpy_para_python(dados_sessao)
        
        self.perfil_atual['sessoes'].append(dados_convertidos)
        self.perfil_atual['estatisticas']['total_sessoes'] += 1
        self.perfil_atual['estatisticas']['tempo_total_uso'] += dados_convertidos.get('duracao', 0)
        self.perfil_atual['estatisticas']['alertas_gerados'] += dados_convertidos.get('total_alertas', 0)
        
# Atualizar HistÓRicos expandido
        if 'EAR_medio' in dados_convertidos:
            self.perfil_atual['estatisticas']['ear_medio_historico'].append(dados_convertidos['EAR_medio'])
        if 'PERCLOS_medio' in dados_convertidos:
            self.perfil_atual['estatisticas']['perclos_medio_historico'].append(dados_convertidos['PERCLOS_medio'])
        if 'BLINK_RATE_medio' in dados_convertidos:
            self.perfil_atual['estatisticas']['blink_rate_medio_historico'].append(dados_convertidos['BLINK_RATE_medio'])
        if 'HEAD_STABILITY_medio' in dados_convertidos:
            self.perfil_atual['estatisticas']['head_stability_medio_historico'].append(dados_convertidos['HEAD_STABILITY_medio'])
        if 'total_bocejos_sessao' in dados_convertidos:
            self.perfil_atual['estatisticas']['total_bocejos_historico'].append(dados_convertidos['total_bocejos_sessao'])
        if 'total_blinks_sessao' in dados_convertidos:
            self.perfil_atual['estatisticas']['total_blinks_historico'].append(dados_convertidos['total_blinks_sessao'])
        
# Manter apenas últimas 50 sessões
        if len(self.perfil_atual['sessoes']) > 50:
            self.perfil_atual['sessoes'] = self.perfil_atual['sessoes'][-50:]
        
# Manter HistÓRicos Com Tamanho Limitado
        for chave in ['ear_medio_historico', 'perclos_medio_historico', 'blink_rate_medio_historico', 
                      'head_stability_medio_historico', 'total_bocejos_historico', 'total_blinks_historico']:
            if len(self.perfil_atual['estatisticas'][chave]) > 100:
                self.perfil_atual['estatisticas'][chave] = self.perfil_atual['estatisticas'][chave][-100:]
        
        self.salvar_perfil()
    
    def get_thresholds(self):
        """Retorna thresholds personalizados expandidos"""
        if self.perfil_atual:
            return self.perfil_atual['thresholds_personalizados']
        else:
# thresholds padrão expandidos
            return {
                'ear_sonolento': 0.25,
                'ear_critico': 0.20,
                'perclos_aviso': 30.0,
                'perclos_atencao': 45.0,
                'perclos_alerta': 60.0,
                'perclos_critico': 80.0,
                'tempo_olhos_fechados_aviso': 1.5,
                'tempo_olhos_fechados_critico': 3.0,
                'mar_bocejo_moderado': 0.30,
                'mar_bocejo_extremo': 0.40,
                'duracao_bocejo_minima': 1.0,
                'duracao_bocejo_critica': 3.0,
                'blink_rate_muito_baixa': 3.0,
                'blink_rate_baixa': 5.0,
                'blink_rate_alta': 25.0,
                'blink_rate_muito_alta': 30.0,
                'blink_rate_normal_min': 8.0,
                'blink_rate_normal_max': 22.0,
                'head_stability_leve': 7.0,
                'head_stability_moderada': 10.0,
                'head_stability_alta': 15.0,
                'microsleep_perclos_threshold': 80.0,
                'microsleep_blink_rate_threshold': 2.0,
                'fadiga_severa_indicators_required': 3,
                'eventos_simultaneos_threshold': 2,
                'max_bocejos_sessao': 2,
                'max_bocejos_critico': 3
            }
    
    def get_estatisticas_avancadas(self):
        """NOVA FUNÇÃO: Retorna estatísticas detalhadas do perfil """
        if not self.perfil_atual:
            return {}
        
        stats = self.perfil_atual['estatisticas']
        
        resumo = {
            'total_sessoes': stats.get('total_sessoes', 0),
            'tempo_total_minutos': stats.get('tempo_total_uso', 0) / 60,
            'alertas_por_sessao': stats.get('alertas_gerados', 0) / max(1, stats.get('total_sessoes', 1)),
        }
        
        # Médias históricas
        if stats.get('ear_medio_historico'):
            resumo['ear_medio_geral'] = np.mean(stats['ear_medio_historico'])
        if stats.get('blink_rate_medio_historico'):
            resumo['blink_rate_medio_geral'] = np.mean(stats['blink_rate_medio_historico'])
        if stats.get('total_bocejos_historico'):
            resumo['bocejos_por_sessao'] = np.mean(stats['total_bocejos_historico'])
        
        return resumo
    
    def listar_usuarios(self):
        """Lista todos os usuários disponíveis"""
        usuarios = []
        for arquivo in self.diretorio.glob("*.json"):
            nome = arquivo.stem
            usuarios.append(nome)
        return sorted(usuarios)

print("✅ GerenciadorPerfis EXPANDIDO criado (JSON-safe + métricas avançadas)")

✅ GerenciadorPerfis EXPANDIDO criado (JSON-safe + métricas avançadas)


In [17]:
# sistema de alertas com suavização temporal e alerta sonoro
class SistemaAlertas:
    def __init__(self):
        self.niveis = {
            'normal': {'cor': (0, 255, 0), 'texto': 'NORMAL', 'prioridade': 0, 'som': None},
            'aviso': {'cor': (0, 255, 255), 'texto': 'AVISO', 'prioridade': 1, 'som': 'beep_suave'},
            'atencao': {'cor': (0, 165, 255), 'texto': 'ATENÇÃO', 'prioridade': 2, 'som': 'beep_medio'},
            'alerta': {'cor': (0, 100, 255), 'texto': 'ALERTA', 'prioridade': 3, 'som': 'beep_forte'},
            'critico': {'cor': (0, 0, 255), 'texto': 'CRÍTICO', 'prioridade': 4, 'som': 'alarme_critico'}
        }
            
        self.historico_alertas = deque(maxlen=100)
        self.ultimo_alerta = None
        self.tempo_ultimo_alerta = 0
        self.contador_alertas = defaultdict(int)
        
# sistema de suavização temporal
        self.buffer_scores = deque(maxlen=15)  # Últimos 15 frames (~0.5s a 30fps)
        self.buffer_niveis = deque(maxlen=10)  # Últimos 10 níveis
        self.nivel_atual = 'normal'
        self.tempo_ultimo_mudanca = 0
        self.frames_consecutivos_mesmo_nivel = 0
        self.scores_suavizados = deque(maxlen=30)  # 1 segundo de scores suavizados
        
# parâmetros de suavização
        self.consenso_necessario = 6
# Frames consecutivos para mudar nível
        self.cooldown_mudanca = 2.0
# Segundos mínimo entre mudanças
        self.threshold_score_mudanca = 10  # Diferença mínima para forçar mudança
        self.filtro_piscada_frames = 5    # Ignorar EAR baixo por poucos frames
        
# sistema de alerta sonoro
        self.alerta_sonoro_habilitado = True
        self.volume_alerta = 0.7  # Volume de 0.0 a 1.0
        self.cooldown_som = 3.0   # Segundos mínimo entre alertas sonoros
        self.ultimo_som_tempo = 0
        self.ultimo_som_nivel = 'normal'
        
# Inicializar sistema de som
        self.inicializar_sistema_som()
    def inicializar_sistema_som(self):
        """NOVO: Inicializa sistema de som com pygame """
        try:
            import pygame
            pygame.mixer.init(frequency=22050, size=-16, channels=2, buffer=512)
            self.pygame_disponivel = True
            print("🔊 Sistema de alerta sonoro inicializado com pygame")
            
            
# Criar sons sintetizados para diferentes níveis
            self.criar_sons_alerta()
            
            
        except ImportError:
            self.pygame_disponivel = False
            print("⚠️ pygame não encontrado, usando beep do sistema")
        except Exception as e:
            self.pygame_disponivel = False
            print(f"⚠️ Erro ao inicializar pygame: {str(e)[:50]}")
    def criar_sons_alerta(self):
        """NOVO: Cria sons sintetizados para alertas """
        if not self.pygame_disponivel:
            return
        
        try:
            import pygame
            import numpy as np
            
            self.sons = {}
            sample_rate = 22050
            
# Beep suave (Aviso)
            freq = 800
            duration = 0.2
            frames = int(duration * sample_rate)
            arr = 4096 * np.sin(2 * np.pi * freq * np.linspace(0, duration, frames))
# Envelope para suavizar
            envelope = np.concatenate([
                np.linspace(0, 1, frames//4),
                np.ones(frames//2),
                np.linspace(1, 0, frames//4)
            ])
            arr = (arr * envelope * 32767 * 0.3).astype(np.int16)  # Volume baixo
            self.sons['beep_suave'] = pygame.sndarray.make_sound(arr)
            
# Beep médio (AtenÇÃO)
            freq = 1000
            duration = 0.3
            frames = int(duration * sample_rate)
            arr = 4096 * np.sin(2 * np.pi * freq * np.linspace(0, duration, frames))
            envelope = np.concatenate([
                np.linspace(0, 1, frames//4),
                np.ones(frames//2),
                np.linspace(1, 0, frames//4)
            ])
            arr = (arr * envelope * 32767 * 0.5).astype(np.int16)  # Volume médio
            self.sons['beep_medio'] = pygame.sndarray.make_sound(arr)
            
# Beep forte (Alerta)
            freq = 1200
            duration = 0.4
            frames = int(duration * sample_rate)
            arr = 4096 * np.sin(2 * np.pi * freq * np.linspace(0, duration, frames))
            envelope = np.concatenate([
                np.linspace(0, 1, frames//4),
                np.ones(frames//2),
                np.linspace(1, 0, frames//4)
            ])
            arr = (arr * envelope * 32767 * 0.6).astype(np.int16)  # Volume alto
            self.sons['beep_forte'] = pygame.sndarray.make_sound(arr)
            
# Alarme crítico (CRÍTico) - 3 beeps rápidos
            freq = 1500
            beep_duration = 0.15
            gap_duration = 0.05
            total_duration = 3 * beep_duration + 2 * gap_duration
            frames = int(total_duration * sample_rate)
            arr = np.zeros(frames)
            
            beep_frames = int(beep_duration * sample_rate)
            gap_frames = int(gap_duration * sample_rate)
            
            beep_arr = 4096 * np.sin(2 * np.pi * freq * np.linspace(0, beep_duration, beep_frames))
            
# 3 beeps
            for i in range(3):
                start = i * (beep_frames + gap_frames)
                end = start + beep_frames
                arr[start:end] = beep_arr
            
            arr = (arr * 32767 * 0.8).astype(np.int16)  # Volume muito alto
            self.sons['alarme_critico'] = pygame.sndarray.make_sound(arr)
            
            
            print("🎵 Sons de alerta criados com sucesso")
            
            
        except Exception as e:
            print(f"⚠️ Erro ao criar sons: {str(e)[:50]}")
            self.sons = {}
    def tocar_alerta_sonoro(self, nivel):
        """NOVO: Toca alerta sonoro baseado no nível """
        if not self.alerta_sonoro_habilitado or nivel == 'normal':
            return
        
        
        tempo_atual = time.time()
        
# Verificar cooldown
        if tempo_atual - self.ultimo_som_tempo < self.cooldown_som:
            return
        
        
# Não repetir som do mesmo nível muito frequentemente
        if (nivel == self.ultimo_som_nivel and
                tempo_atual - self.ultimo_som_tempo < self.cooldown_som * 2):
            return
        
        som_tipo = self.niveis[nivel]['som']
        if not som_tipo:
            return
        
        
        try:
            if self.pygame_disponivel and som_tipo in self.sons:
# Usar pygame
                som = self.sons[som_tipo]
                som.set_volume(self.volume_alerta)
                som.play()
                print(f"🔊 Alerta sonoro: {nivel.upper()} ({som_tipo})")
            else:
# usar beep do sistema como fallback
                self._tocar_beep_linux(nivel)
                
        except Exception as e:
            print(f"⚠️ Erro ao tocar som: {str(e)[:30]}")
        
        self.ultimo_som_tempo = tempo_atual
        self.ultimo_som_nivel = nivel
    def processar_nivel_temporal(self, score_fadiga, nivel_candidato):
        """SISTEMA DE SUAVIZAÇÃO TEMPORAL COM CONSENSO """
        tempo_atual = time.time()
        
# Adicionar score ao buffer
        self.buffer_scores.append(score_fadiga)
        self.buffer_niveis.append(nivel_candidato)
        
# Calcular score suavizado
        if len(self.buffer_scores) >= 5:
            score_suavizado = np.mean(list(self.buffer_scores)[-10:])  # Média dos últimos 10
            self.scores_suavizados.append(score_suavizado)
        else:
            score_suavizado = score_fadiga
        
# Lógica de consenso melhorada
        if len(self.buffer_niveis) >= self.consenso_necessario:
# Contar ocorrências do nível candidato nos últimos frames
            ultimos_niveis = list(self.buffer_niveis)[-self.consenso_necessario:]
            count_novo_nivel = ultimos_niveis.count(nivel_candidato)
            
# Verificar se há consenso
            consenso_atingido = count_novo_nivel >= (self.consenso_necessario - 1)  # 5 de 6
            
# Verificar cooldown
            cooldown_expirou = tempo_atual - self.tempo_ultimo_mudanca >= self.cooldown_mudanca
            
# Permitir mudança crítica imediata
            mudanca_critica = (nivel_candidato == 'critico' and score_fadiga >= 80)
            
# Verificar deterioração significativa
            if len(self.scores_suavizados) >= 10:
                tendencia = np.mean(list(self.scores_suavizados)[-5:]) - np.mean(list(self.scores_suavizados)[-10:-5])
                deterioracao_significativa = tendencia > self.threshold_score_mudanca
            else:
                deterioracao_significativa = False
            
# Decidir se mudar
            if consenso_atingido and (cooldown_expirou or mudanca_critica or deterioracao_significativa):
                nivel_anterior = self.nivel_atual
                self.nivel_atual = nivel_candidato
                self.tempo_ultimo_mudanca = tempo_atual
                self.frames_consecutivos_mesmo_nivel = 0
                
                
# Tocar alerta sonoro Quando Muda NÍVel
                if (self.niveis[nivel_candidato]['prioridade'] > self.niveis[nivel_anterior]['prioridade']):
                    self.tocar_alerta_sonoro(nivel_candidato)
                    
                
                print(f"🔄 Mudança de nível para: {nivel_candidato.upper()} (consenso: {count_novo_nivel}/{self.consenso_necessario})")
                return self.nivel_atual
        return self.nivel_atual if self.nivel_atual else 'normal'
    def determinar_nivel(self, metricas, thresholds):
        """ALGORITMO EXPANDIDO COM SUAVIZAÇÃO: Determina nível baseado em TODAS as métricas """
# Extrair métricas
        perclos = metricas.get('PERCLOS', 0)
        ear = metricas.get('EAR', 1.0)
        mar = metricas.get('MAR', 0.0)
        blink_rate = metricas.get('BLINK_RATE', 15.0)
        head_stability = metricas.get('HEAD_STABILITY', 5.0)
        tempo_olhos_fechados = metricas.get('TEMPO_OLHOS_FECHADOS', 0)
        total_bocejos = metricas.get('TOTAL_BOCEJOS', 0)
        
# Score De Fadiga Calculado Com MÚLtiplas MÉTricas
        score_fadiga = 0
        motivos = []
    
# 🚨 SonolÊNcia CRÍTica AutomÁTica
        sonolencia_critica = False
        tempo_critico = thresholds.get('tempo_olhos_fechados_critico', 5.0)  # padrão 5s se não definido
    
        if tempo_olhos_fechados >= tempo_critico:
            score_fadiga = 100
            motivos.append(f"Olhos fechados críticos ({tempo_olhos_fechados:.1f}s) — sonolência 100%")
            sonolencia_critica = True
        elif perclos >= thresholds.get('perclos_critico', 80):
            score_fadiga = 100
            motivos.append(f"PERCLOS crítico ({perclos:.1f}%) — sonolência 100%")
            sonolencia_critica = True
    
# Se for sonolência crítica, pula os cálculos restantes
        if not sonolencia_critica:
# 1. Perclos (peso 40%)
            if perclos >= thresholds['perclos_critico']:  # 80%
                score_fadiga += 40
                motivos.append(f"PERCLOS crítico ({perclos:.1f}%)")
            elif perclos >= thresholds['perclos_alerta']:  # 60%
                score_fadiga += 30
                motivos.append(f"PERCLOS alto ({perclos:.1f}%)")
            elif perclos >= thresholds['perclos_atencao']:  # 45%
                score_fadiga += 20
                motivos.append(f"PERCLOS moderado ({perclos:.1f}%)")
            elif perclos >= thresholds['perclos_aviso']:  # 30%
                score_fadiga += 10
                motivos.append(f"PERCLOS elevado ({perclos:.1f}%)")
            
# 2. Ear Baixo (peso 25%)
            if ear <= thresholds['ear_critico']:  # Muito baixo
                score_fadiga += 25
                motivos.append(f"EAR crítico ({ear:.3f})")
            elif ear <= thresholds['ear_sonolento']:
                score_fadiga += 15
                motivos.append(f"EAR baixo ({ear:.3f})")
            
# 3. Tempo Olhos Fechados (peso 20%)
            if tempo_olhos_fechados >= 3.0:
                score_fadiga += 20
                motivos.append(f"Olhos fechados {tempo_olhos_fechados:.1f}s")
            elif tempo_olhos_fechados >= 1.5:
                score_fadiga += 10
                motivos.append(f"Olhos fechados {tempo_olhos_fechados:.1f}s")
            
# 4. Bocejos (peso 10%)
            if total_bocejos >= 3:
                score_fadiga += 10
                motivos.append(f"Bocejos frequentes ({total_bocejos})")
                
# 5. Blink Rate Anormal (peso 12%)
            if blink_rate <= 3:
                score_fadiga += 12
                motivos.append(f"Piscadas raras ({blink_rate:.1f}/min)")
            elif blink_rate >= 30:
                score_fadiga += 8
                motivos.append(f"Piscadas excessivas ({blink_rate:.1f}/min)")
            
# 6. Head Stability (peso 8%)
            if head_stability >= 10.0:
                score_fadiga += 8
                motivos.append(f"Cabeça instável ({head_stability:.1f})")
            
# 7. Mar Elevado (peso 5%)
            if mar >= thresholds.get('mar_bocejo', 0.6):
                score_fadiga += 5
                motivos.append(f"Bocejo detectado ({mar:.3f})")
    
# Garantir que score não exceda 100
        score_fadiga = min(100, score_fadiga)
        
# Determinar NÍVel Baseado No Score
        if score_fadiga >= 80:
            nivel_candidato = 'critico'
        elif score_fadiga >= 60:
            nivel_candidato = 'alerta'  
        elif score_fadiga >= 40:
            nivel_candidato = 'atencao'
        elif score_fadiga >= 20:
            nivel_candidato = 'aviso'
        else:
            nivel_candidato = 'normal'
        
# Aplicar suavização temporal
        nivel_final = self.processar_nivel_temporal(score_fadiga, nivel_candidato)
        
        return nivel_final, score_fadiga, motivos

    def processar_alerta(self, nivel, score, motivos):
        """Processa e registra alerta com conversão JSON-safe"""
        tempo_atual = time.time()
        
# conversão para tipos Python
        alerta = {
            'timestamp': float(tempo_atual),
            'nivel': str(nivel),
            'score': float(score),  # Garantir que é float Python
            'motivos': [str(motivo) for motivo in motivos] if motivos else [],
            'info': self.niveis[nivel].copy(),
# InformaÇÕES De suavização
            'score_suavizado': True,
            'frames_consecutivos': self.frames_consecutivos_mesmo_nivel,
            'consenso_frames': len(self.buffer_niveis),
# InformaÇÕES De Som
            'som_tocado': self.alerta_sonoro_habilitado and nivel != 'normal'
        }
        
# Converter dados do alerta para tipos Python
        alerta_convertido = self.converter_numpy_para_python(alerta)
        
        self.historico_alertas.append(alerta_convertido)
        self.contador_alertas[nivel] += 1
        
# Atualizar estado
        self.ultimo_alerta = alerta_convertido
        self.tempo_ultimo_alerta = tempo_atual
        
        return alerta_convertido
    def get_estatisticas(self):
        """Retorna estatísticas dos alertas"""
        total = sum(self.contador_alertas.values())
        if total == 0:
            return {}
        
        estatisticas = {
            'total_alertas': int(total),  # Garantir int Python
            'por_nivel': {k: int(v) for k, v in self.contador_alertas.items()},
            'percentuais': {nivel: float((count/total)*100)
                             for nivel, count in self.contador_alertas.items()},
            'ultimo_alerta': self.ultimo_alerta,
# estatísticas De suavização
            'score_suavizado_medio': float(np.mean(self.scores_suavizados)) if self.scores_suavizados else 0.0,
            'nivel_atual': self.nivel_atual,
            'tempo_no_nivel_atual': float(time.time() - self.tempo_ultimo_mudanca)
        }
        
        return estatisticas
    def converter_numpy_para_python(self, obj):
        """NOVO: Converte tipos NumPy para tipos Python nativos """
        if isinstance(obj, dict):
            return {k: self.converter_numpy_para_python(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [self.converter_numpy_para_python(item) for item in obj]
        elif isinstance(obj, tuple):
            return tuple(self.converter_numpy_para_python(item) for item in obj)
        elif isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return obj
    def configurar_som(self, habilitado=None, volume=None, cooldown=None):
        """NOVO: Configura sistema de som """
        if habilitado is not None:
            self.alerta_sonoro_habilitado = habilitado
            print(f"🔊 Alerta sonoro: {'HABILITADO' if habilitado else 'DESABILITADO'}")
            
            
        if volume is not None:
            self.volume_alerta = max(0.0, min(1.0, volume))
            print(f"🔊 Volume do alerta: {self.volume_alerta:.1f}")
            
            
        if cooldown is not None:
            self.cooldown_som = max(1.0, cooldown)
            print(f"🔊 Cooldown do som: {self.cooldown_som:.1f}s")
    def testar_alertas_sonoros(self):
        """NOVO: Testa todos os alertas sonoros """
        print("🎵 Testando alertas sonoros...")
        time.sleep(1)
        
        
        for nivel in ['aviso', 'atencao', 'alerta', 'critico']:
            print(f"  Testando: {nivel.upper()}")
            self.tocar_alerta_sonoro(nivel)
            time.sleep(2)  # Pausa entre testes
            
        print("✅ Teste de alertas sonoros concluído")
    def reset_sistema_suavizacao(self):
        """NOVO: Reseta sistema de suavização (útil para recalibração) """
        self.buffer_scores.clear()
        self.buffer_niveis.clear()
        self.scores_suavizados.clear()
        self.nivel_atual = 'normal'
        self.frames_consecutivos_mesmo_nivel = 0
        self.tempo_ultimo_mudanca = 0
        self.ultimo_som_tempo = 0
        self.ultimo_som_nivel = 'normal'
        print("🔄 Sistema de suavização resetado")
    def _tocar_beep_linux(self, nivel):
        """NOVO: Método específico para beep no Linux """
        import subprocess
        import tempfile
        import os
        import wave
        import math
        import struct
        
# Configurar frequência e duração baseado no nível
        configs = {
            'aviso': (800, 0.2),
            'atencao': (1000, 0.3),
            'alerta': (1200, 0.4),
            'critico': (1500, 0.5)
        }
        
        frequency, duration = configs.get(nivel, (1000, 0.3))
        
        try:
# Método: paplay com arquivo Wav temporário
            with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
                filename = tmp.name
            
# Criar arquivo Wav
            sample_rate = 22050
            with wave.open(filename, 'w') as wav_file:
                wav_file.setnchannels(1)  # Mono
                wav_file.setsampwidth(2)  # 2 bytes per sample
                wav_file.setframerate(sample_rate)
                
# Gerar beep crítico com múltiplos beeps
                if nivel == 'critico':
# 3 beeps rápidos
                    for beep_num in range(3):
                        for i in range(int(sample_rate * 0.15)):
                            value = int(16383 * math.sin(2 * math.pi * frequency * i / sample_rate))
                            wav_file.writeframes(struct.pack('<h', value))
# Pausa entre beeps
                        if beep_num < 2:
                            for i in range(int(sample_rate * 0.05)):
                                wav_file.writeframes(struct.pack('<h', 0))
                else:
# Beep único
                    for i in range(int(sample_rate * duration)):
                        value = int(16383 * math.sin(2 * math.pi * frequency * i / sample_rate))
                        wav_file.writeframes(struct.pack('<h', value))
            
# Tocar com paplay
            result = subprocess.run(['paplay', filename],
                                  capture_output=True, timeout=duration + 1)
            os.unlink(filename)
            
            if result.returncode == 0:
                print(f"🔊 Beep Linux: {nivel.upper()} ({frequency}Hz)")
                return
        
        except Exception as e1:
            try:
# Cleanup arquivo se deu erro
                if 'filename' in locals():
                    os.unlink(filename)
            except:
                pass
            
            print(f"🔊 Beep do sistema: {nivel.upper()} (fallback - {str(e1)[:30]})")
            print('\a')  # Tentar mesmo assim

print("✅ SistemaAlertas EXPANDIDO criado (com suavização temporal e alerta sonoro)")


✅ SistemaAlertas EXPANDIDO criado (com suavização temporal e alerta sonoro)


In [18]:
# demonstração e configuração do sistema de alerta sonoro
# Exemplo de uso do sistema de alerta sonoro

print("🎵 Sistema de Alerta Sonoro - Demonstração")
print("=" * 50)

# Criar sistema de alertas (já feito automaticamente na inicialização)
# sistema_teste = Sistemaalertas()

# Configurações disponíveis:
print("📋 Configurações disponíveis:")
print("1. Habilitar/Desabilitar alertas sonoros")
print("2. Ajustar volume (0.0 a 1.0)")
print("3. Configurar cooldown entre alertas")
print("4. Testar todos os tipos de alerta")

print("🔊 Tipos de alerta sonoro:")
print("• AVISO: Beep suave (800Hz, 0.2s)")
print("• ATENÇÃO: Beep médio (1000Hz, 0.3s)")
print("• ALERTA: Beep forte (1200Hz, 0.4s)")
print("• CRÍTICO: Alarme sequencial (1500Hz, 3 beeps)")

print("⚙️ Para configurar o sistema de som:")
print("# detector.sistema_alertas.configurar_som(habilitado=True, volume=0.7, cooldown=3.0)")

print("🧪 Para testar os alertas:")
print("# detector.sistema_alertas.testar_alertas_sonoros()")

print("📊 Recursos implementados:")
print("✅ Sons sintetizados com pygame")
print("✅ Fallback para beep do sistema")
print("✅ Cooldown para evitar spam sonoro")
print("✅ Volume configurável")
print("✅ Integração com temporal smoothing")
print("✅ Diferentes sons para cada nível")

print("💡 Dicas de uso:")
print("• O sistema toca automaticamente quando o nível de alerta muda")
print("• Sons só tocam quando há deterioração (nível aumenta)")
print("• Cooldown padrão: 3 segundos entre alertas")
print("• Volume padrão: 0.7 (70%)")
print("• Se pygame não estiver disponível, usa beep do sistema")

print("🚀 Sistema pronto para uso!")

🎵 Sistema de Alerta Sonoro - Demonstração
📋 Configurações disponíveis:
1. Habilitar/Desabilitar alertas sonoros
2. Ajustar volume (0.0 a 1.0)
3. Configurar cooldown entre alertas
4. Testar todos os tipos de alerta
🔊 Tipos de alerta sonoro:
• AVISO: Beep suave (800Hz, 0.2s)
• ATENÇÃO: Beep médio (1000Hz, 0.3s)
• ALERTA: Beep forte (1200Hz, 0.4s)
• CRÍTICO: Alarme sequencial (1500Hz, 3 beeps)
⚙️ Para configurar o sistema de som:
# detector.sistema_alertas.configurar_som(habilitado=True, volume=0.7, cooldown=3.0)
🧪 Para testar os alertas:
# detector.sistema_alertas.testar_alertas_sonoros()
📊 Recursos implementados:
✅ Sons sintetizados com pygame
✅ Fallback para beep do sistema
✅ Cooldown para evitar spam sonoro
✅ Volume configurável
✅ Integração com temporal smoothing
✅ Diferentes sons para cada nível
💡 Dicas de uso:
• O sistema toca automaticamente quando o nível de alerta muda
• Sons só tocam quando há deterioração (nível aumenta)
• Cooldown padrão: 3 segundos entre alertas
• Volume pad

In [19]:
# Analisador de sessão avançado
class AnalisadorSessao:
    def __init__(self):
        self.dados_sessao = {
            'inicio': time.time(),
            'metricas_temporais': deque(maxlen=1800),  # 1 minuto a 30fps
            'alertas': [],
            'estatisticas': defaultdict(list)
        }
        
    def adicionar_metrica(self, metricas, nivel_alerta):
        """Adiciona métricas do frame atual"""
        timestamp = time.time()
        
        entrada = {
            'timestamp': timestamp,
            'metricas': metricas.copy(),
            'nivel_alerta': nivel_alerta
        }
        
        self.dados_sessao['metricas_temporais'].append(entrada)
        
# Atualizar estatísticas
        for chave, valor in metricas.items():
            if isinstance(valor, (int, float)):
                self.dados_sessao['estatisticas'][chave].append(valor)
    
    def adicionar_alerta(self, alerta):
        """Adiciona alerta à sessão"""
        self.dados_sessao['alertas'].append(alerta)
    
    def analisar_tendencias(self, janela_minutos=2):
        """Analisa tendências nos últimos minutos"""
        if len(self.dados_sessao['metricas_temporais']) < 60:  # Pelo menos 2 segundos
            return {'tendencia': 'insuficiente', 'dados': []}
        
        tempo_limite = time.time() - (janela_minutos * 60)
        dados_recentes = [entry for entry in self.dados_sessao['metricas_temporais'] 
                         if entry['timestamp'] > tempo_limite]
        
        if len(dados_recentes) < 30:
            return {'tendencia': 'insuficiente', 'dados': []}
        
# Analisar Perclos
        perclos_valores = [entry['metricas'].get('PERCLOS', 0) for entry in dados_recentes]
        ear_valores = [entry['metricas'].get('EAR', 0.3) for entry in dados_recentes]
        
# Calcular tendências
        x = np.arange(len(perclos_valores))
        
        tendencia_perclos = 'estavel'
        tendencia_ear = 'estavel'
        
        if len(perclos_valores) > 10:
            try:
                slope_perclos, _, _, _, _ = stats.linregress(x, perclos_valores)
                slope_ear, _, _, _, _ = stats.linregress(x, ear_valores)
                
                if slope_perclos > 0.5:
                    tendencia_perclos = 'deteriorando'
                elif slope_perclos < -0.5:
                    tendencia_perclos = 'melhorando'
                
                if slope_ear < -0.001:
                    tendencia_ear = 'deteriorando'
                elif slope_ear > 0.001:
                    tendencia_ear = 'melhorando'
            except:
                pass
        
# Análise de padrões de alerta
        alertas_recentes = [entry['nivel_alerta'] for entry in dados_recentes]
        alertas_serios = sum(1 for a in alertas_recentes if a in ['alerta', 'critico'])
        taxa_alertas = (alertas_serios / len(alertas_recentes)) * 100
        
        return {
            'tendencia_perclos': tendencia_perclos,
            'tendencia_ear': tendencia_ear,
            'perclos_medio': np.mean(perclos_valores),
            'ear_medio': np.mean(ear_valores),
            'taxa_alertas': taxa_alertas,
            'total_pontos': len(dados_recentes)
        }
    
    def converter_numpy_para_python(self, obj):
        """Converte tipos NumPy para tipos Python nativos (JSON serializável)"""
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, dict):
            return {key: self.converter_numpy_para_python(value) for key, value in obj.items()}
        elif isinstance(obj, list):
            return [self.converter_numpy_para_python(item) for item in obj]
        elif isinstance(obj, defaultdict):
            return {key: self.converter_numpy_para_python(value) for key, value in obj.items()}
        else:
            return obj
    
    def gerar_relatorio(self):
        """Gera relatório completo da sessão"""
        duracao = time.time() - self.dados_sessao['inicio']
        
        relatorio = {
            'duracao': float(duracao),
            'timestamp': datetime.now().isoformat(),
            'total_alertas': len(self.dados_sessao['alertas']),
            'alertas_por_nivel': defaultdict(int)
        }
        
# Contar alertas por nível
        for alerta in self.dados_sessao['alertas']:
            relatorio['alertas_por_nivel'][alerta['nivel']] += 1
        
# estatísticas das métricas - conversão para tipos Python
        for metrica, valores in self.dados_sessao['estatisticas'].items():
            if valores:
                relatorio[f'{metrica}_medio'] = float(np.mean(valores))
                relatorio[f'{metrica}_min'] = float(np.min(valores))
                relatorio[f'{metrica}_max'] = float(np.max(valores))
                relatorio[f'{metrica}_std'] = float(np.std(valores))
        
        relatorio_convertido = self.converter_numpy_para_python(relatorio)
        
        return relatorio_convertido

print("✅ AnalisadorSessao criado (JSON-safe)")

✅ AnalisadorSessao criado (JSON-safe)


In [20]:
# Detector de Fadiga inteligente Principal - Com Regras AvanÇAdas
class DetectorFadigaInteligente:
    def __init__(self, gerenciador_perfis):
# Mediapipe
        self.mp_face_mesh = mp.solutions.face_mesh
        self.face_mesh = self.mp_face_mesh.FaceMesh(
            static_image_mode=False,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.7,
            min_tracking_confidence=0.5
        )
        
# Landmarks
        self.OLHO_ESQUERDO = [362, 385, 387, 263, 373, 380]
        self.OLHO_DIREITO = [33, 160, 158, 133, 153, 144]
        self.BOCA_LANDMARKS = [13, 14, 12, 15, 16, 18, 61, 291]
        
# Landmarks Para Head Stability
        self.PONTOS_CABECA = [10, 151, 9, 8, 168, 6, 197, 195, 5, 4, 1, 2]  # Contorno da face
        self.PONTO_NARIZ = [1]  # Ponta do nariz para referência central
        
# Sistemas auxiliares
        self.gerenciador_perfis = gerenciador_perfis
        self.sistema_alertas = SistemaAlertas()
        self.analisador_sessao = AnalisadorSessao()
        
# Buffers para análise temporal expandido
        self.buffer_ear = deque(maxlen=300)  # 10 segundos
        self.buffer_mar = deque(maxlen=150)  # 5 segundos
        self.buffer_blink = deque(maxlen=900) # 30 segundos
        self.buffer_sequencia = deque(maxlen=90) # Para modelo ML
        
# Novos Buffers Para MÉTricas AvanÇAdas
        self.buffer_head_positions = deque(maxlen=300)  # 10 segundos de posições da cabeça
        self.buffer_blink_events = deque(maxlen=60)     # 2 minutos de eventos de piscada
        self.buffer_yawn_events = deque(maxlen=30)      # 1 minuto de eventos de bocejo
        
# Estado temporal expandido
        self.tempo_olhos_fechados = 0
        self.ultimo_tempo = time.time()
        self.frames_processados = 0
        self.calibrando = False
        self.dados_calibracao = {'ear': [], 'mar': [], 'blink': [], 'head_stability': []}
        
# Estados Para DetecÇÃO De Eventos
        self.ultimo_ear = 0.3
        self.em_blink = False
        self.tempo_ultimo_blink = 0
        self.em_bocejo = False
        self.tempo_ultimo_bocejo = 0
        self.posicao_cabeca_anterior = None
        
# Carregar modelo Ml como backup
        self.carregar_modelo()
        
        print("🧠 Detector Inteligente Inicializado - Regras Avançadas")
        
    def carregar_modelo(self):
        """Carrega modelo XGBoost como backup"""
        try:
            modelo_path = Path('modelos_xgb')
            if modelo_path.exists():
# Definir extrator sem pesos
                class ExtratorFeatures:
                    def __init__(self):
                        nomes_sinais = ['PERCLOS', 'MAR', 'BLINK_RATE', 'HEAD_STABILITY']
                        nomes_stats = ['mean', 'std', 'median', 'min', 'max', 'range', 'q25', 'q75', 'trend', 'zcr', 'autocorr']
                        self.feature_names = []
                        for sinal in nomes_sinais:
                            for stat in nomes_stats:
                                self.feature_names.append(f"{sinal}_{stat}")
                    
                    def trend_slope(self, sinal):
                        if len(sinal) < 2: return 0.0
                        x = np.arange(len(sinal))
                        try:
                            slope, _, _, _, _ = stats.linregress(x, sinal)
                            return slope if not np.isnan(slope) else 0.0
                        except: return 0.0
                    
                    def zero_crossing_rate(self, sinal):
                        if len(sinal) < 2: return 0.0
                        mean_centered = sinal - np.mean(sinal)
                        crossings = np.sum(np.diff(np.sign(mean_centered)) != 0)
                        return crossings / len(sinal)
                    
                    def autocorr_lag1(self, sinal):
                        if len(sinal) < 3: return 0.0
                        try:
                            corr = np.corrcoef(sinal[:-1], sinal[1:])[0, 1]
                            return corr if not np.isnan(corr) else 0.0
                        except: return 0.0
                    
                    def extrair_features_sinal(self, sinal):
                        features = [
                            np.mean(sinal), np.std(sinal), np.median(sinal),
                            np.min(sinal), np.max(sinal), np.ptp(sinal),
                            np.percentile(sinal, 25), np.percentile(sinal, 75),
                            self.trend_slope(sinal), self.zero_crossing_rate(sinal), self.autocorr_lag1(sinal)
                        ]
                        return features
                    
                    def transform(self, X_sequences):
                        n_samples = X_sequences.shape[0]
                        n_features = len(self.feature_names)
                        X_features = np.zeros((n_samples, n_features))
                        for i in range(n_samples):
                            sequence = X_sequences[i]
                            sample_features = []
                            for signal_idx in range(4):
                                sinal = sequence[:, signal_idx]
                                features_sinal = self.extrair_features_sinal(sinal)
                                sample_features.extend(features_sinal)
                            X_features[i] = sample_features
                        return X_features
                
                self.extrator = ExtratorFeatures()
                self.scaler = joblib.load(modelo_path / 'scaler.joblib')
                self.modelo = joblib.load(modelo_path / 'modelo_xgb.joblib')
                self.modelo_disponivel = True
                print("🤖 Modelo ML carregado")
            else:
                self.modelo_disponivel = False
        except Exception as e:
            self.modelo_disponivel = False
            print(f"⚠️ Modelo ML indisponível: {str(e)[:50]}")
    
    def iniciar_calibracao(self, duracao_segundos=30):
        """Inicia processo de calibração"""
        self.calibrando = True
        self.dados_calibracao = {'ear': [], 'mar': [], 'blink': [], 'head_stability': []}
        self.tempo_calibracao_fim = time.time() + duracao_segundos
        print(f"🎯 Iniciando calibração por {duracao_segundos}s...")
        print("💡 Mantenha expressão relaxada e natural")
        print("💡 Evite movimentos excessivos da cabeça")
    
    def calcular_ear(self, landmarks_olho):
        """Calcula EAR"""
        v1 = euclidean(landmarks_olho[1], landmarks_olho[5])
        v2 = euclidean(landmarks_olho[2], landmarks_olho[4])
        h = euclidean(landmarks_olho[0], landmarks_olho[3])
        if h < 1e-6: return 0.0
        ear = (v1 + v2) / (2.0 * h)
        return max(0.0, min(1.0, ear))
    
    def calcular_mar(self, landmarks_px):
        """Calcula MAR corrigido"""
        try:
            v1 = euclidean(landmarks_px[13], landmarks_px[14])
            v2 = euclidean(landmarks_px[12], landmarks_px[15])
            v3 = euclidean(landmarks_px[16], landmarks_px[18])
            h = euclidean(landmarks_px[61], landmarks_px[291])
            if h < 1e-6: return 0.0
            mar = (v1 + v2 + v3) / (3.0 * h)
            return max(0.0, min(2.0, mar))
        except: return 0.0
    
    def calcular_head_stability(self, landmarks_px):
        """NOVA FUNÇÃO: Calcula estabilidade da cabeça """
        try:
# Calcular posição central da cabeça (média dos pontos de referência)
            pontos_cabeca = [landmarks_px[i] for i in self.PONTOS_CABECA]
            centro_x = np.mean([p[0] for p in pontos_cabeca])
            centro_y = np.mean([p[1] for p in pontos_cabeca])
            
            posicao_atual = (centro_x, centro_y)
            
            if self.posicao_cabeca_anterior is None:
                self.posicao_cabeca_anterior = posicao_atual
                return 0.0
            
# Calcular movimento desde último frame
            distancia_movimento = euclidean(posicao_atual, self.posicao_cabeca_anterior)
            self.posicao_cabeca_anterior = posicao_atual
            
# Adicionar ao buffer de posições
            self.buffer_head_positions.append(distancia_movimento)
            
# Calcular instabilidade (desvio padrão dos movimentos)
            if len(self.buffer_head_positions) >= 30:  # Pelo menos 1 segundo
                instabilidade = np.std(list(self.buffer_head_positions)[-30:])
                return min(100.0, instabilidade)  # Limitar a 100
            
            return 0.0
        except:
            return 0.0
    
    def detectar_blink(self, ear_atual):
        """NOVA FUNÇÃO: Detecta eventos de piscada """
        threshold_blink = 0.25  # Threshold para detecção de piscada
        tempo_atual = time.time()
        
# Detectar início de piscada (Ear cai abaixo do threshold)
        if not self.em_blink and ear_atual <= threshold_blink:
            self.em_blink = True
            return False
        
# Detectar fim de piscada (Ear volta acima do threshold)
        elif self.em_blink and ear_atual > threshold_blink:
            self.em_blink = False
            
# Registrar evento de piscada
            tempo_desde_ultimo = tempo_atual - self.tempo_ultimo_blink
            if tempo_desde_ultimo > 0.1:  # Mínimo 100ms entre piscadas
                self.buffer_blink_events.append(tempo_atual)
                self.tempo_ultimo_blink = tempo_atual
                return True
        
        return False
    
    def calcular_blink_rate(self):
        """NOVA FUNÇÃO: Calcula taxa de piscadas por minuto """
        tempo_atual = time.time()
        tempo_limite = tempo_atual - 60  # Últimos 60 segundos
        
# Filtrar piscadas dos últimos 60 segundos
        blinks_recentes = [t for t in self.buffer_blink_events if t > tempo_limite]
        
# Taxa de piscadas por minuto
        blink_rate = len(blinks_recentes)
        return min(60.0, blink_rate)  # Máximo realista de 60 piscadas/min
    
    def detectar_bocejo(self, mar_atual):
        """NOVA FUNÇÃO: Detecta eventos de bocejo """
        threshold_bocejo = 0.35  # Threshold para detecção de bocejo
        tempo_atual = time.time()
        duracao_minima_bocejo = 1.0  # Bocejo deve durar pelo menos 1 segundo
        
# Detectar início de bocejo
        if not self.em_bocejo and mar_atual >= threshold_bocejo:
            self.em_bocejo = True
            self.inicio_bocejo = tempo_atual
            return False, 0.0
        
# Se está em bocejo, calcular duração
        elif self.em_bocejo:
            duracao_bocejo = tempo_atual - self.inicio_bocejo
            
# Bocejo terminou (Mar caiu) e durou tempo suficiente
            if mar_atual < threshold_bocejo and duracao_bocejo >= duracao_minima_bocejo:
                self.em_bocejo = False
                
# Registrar evento de bocejo
                tempo_desde_ultimo = tempo_atual - self.tempo_ultimo_bocejo
                if tempo_desde_ultimo > 5.0:  # Mínimo 5 segundos entre bocejos
                    self.buffer_yawn_events.append(tempo_atual)
                    self.tempo_ultimo_bocejo = tempo_atual
                    return True, duracao_bocejo
            
# Bocejo muito longo, considerar falso positivo
            elif duracao_bocejo > 10.0:
                self.em_bocejo = False
            
            return False, duracao_bocejo
        
        return False, 0.0
    
    def calcular_perclos(self, janela_segundos=3):
        """Calcula PERCLOS"""
        if len(self.buffer_ear) < 30: return 0.0
        
# Pegar dados da janela especificada
        janela_frames = min(len(self.buffer_ear), janela_segundos * 30)
        ears_recentes = list(self.buffer_ear)[-janela_frames:]
        
# Usar threshold personalizado se disponível
        thresholds = self.gerenciador_perfis.get_thresholds()
        threshold_fechado = thresholds.get('ear_sonolento', 0.25)
        
        frames_fechados = sum(1 for ear in ears_recentes if ear <= threshold_fechado)
        perclos = (frames_fechados / len(ears_recentes)) * 100
        return min(100.0, perclos)
    
    def processar_frame(self, frame):
        """Processamento principal do frame - EXPANDIDO COM NOVAS MÉTRICAS"""
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resultados = self.face_mesh.process(frame_rgb)
        
        self.frames_processados += 1
        tempo_atual = time.time()
        dt = tempo_atual - self.ultimo_tempo
        self.ultimo_tempo = tempo_atual
        
# Verificar se ainda está calibrando
        if self.calibrando:
            if tempo_atual > self.tempo_calibracao_fim:
                self.finalizar_calibracao()
        
        if not resultados.multi_face_landmarks:
            return frame, "Nenhum rosto detectado", "normal", 0.0, {}
        
        landmarks = resultados.multi_face_landmarks[0]
        h, w = frame.shape[:2]
        landmarks_px = [(lm.x * w, lm.y * h) for lm in landmarks.landmark]
        
# Calcular métricas básicas
        olho_esquerdo = [landmarks_px[i] for i in self.OLHO_ESQUERDO]
        olho_direito = [landmarks_px[i] for i in self.OLHO_DIREITO]
        ear_esquerdo = self.calcular_ear(olho_esquerdo)
        ear_direito = self.calcular_ear(olho_direito)
        ear_medio = (ear_esquerdo + ear_direito) / 2.0
        
        mar = self.calcular_mar(landmarks_px)
        
# Calcular Novas MÉTricas AvanÇAdas
        head_stability = self.calcular_head_stability(landmarks_px)
        
# Detectar eventos
        blink_detectado = self.detectar_blink(ear_medio)
        bocejo_detectado, duracao_bocejo = self.detectar_bocejo(mar)
        
# Calcular taxas
        blink_rate = self.calcular_blink_rate()
        
# Atualizar buffers
        self.buffer_ear.append(ear_medio)
        self.buffer_mar.append(mar)
        
# Calcular Perclos
        perclos = self.calcular_perclos()
        
# Atualizar tempo de olhos fechados
        thresholds = self.gerenciador_perfis.get_thresholds()
        if ear_medio <= thresholds['ear_sonolento']:
            self.tempo_olhos_fechados += dt
        else:
            self.tempo_olhos_fechados = 0
        
# Se calibrando, coletar dados
        if self.calibrando:
            self.dados_calibracao['ear'].append(ear_medio)
            self.dados_calibracao['mar'].append(mar)
            self.dados_calibracao['blink'].append(blink_rate if blink_rate > 0 else 15.0)
            self.dados_calibracao['head_stability'].append(head_stability)
            
# Desenhar informações de calibração
            frame_anotado = self.desenhar_calibracao(frame, tempo_atual)
            return frame_anotado, "Calibrando...", "normal", 1.0, {'EAR': ear_medio, 'PERCLOS': perclos}
        
# MÉTricas Completas expandidas
        metricas = {
            'EAR': ear_medio,
            'PERCLOS': perclos,
            'MAR': mar,
            'BLINK_RATE': blink_rate,
            'HEAD_STABILITY': head_stability,
            'tempo_olhos_fechados': self.tempo_olhos_fechados,
            'frames_processados': self.frames_processados,
            'blink_detectado': blink_detectado,
            'bocejo_detectado': bocejo_detectado,
            'duracao_bocejo': duracao_bocejo,
            'total_bocejos': len(self.buffer_yawn_events),
            'total_blinks': len(self.buffer_blink_events)
        }
        
# Determinar nível de alerta
        nivel_alerta, score_fadiga, motivos = self.sistema_alertas.determinar_nivel(metricas, thresholds)
        
# Processar alerta
        alerta = self.sistema_alertas.processar_alerta(nivel_alerta, score_fadiga, motivos)
        
# Adicionar à análise de sessão
        self.analisador_sessao.adicionar_metrica(metricas, nivel_alerta)
        if nivel_alerta != 'normal':
            self.analisador_sessao.adicionar_alerta(alerta)
        
# Análise de tendências
        tendencias = self.analisador_sessao.analisar_tendencias()
        
# Desenhar informações
        frame_anotado = self.desenhar_interface_avancada(frame, landmarks_px, alerta,
                                                         metricas, tendencias, thresholds)
        
        return frame_anotado, alerta['info']['texto'], nivel_alerta, score_fadiga/100, metricas
    
    def finalizar_calibracao(self):
        """Finaliza processo de calibração expandido"""
        self.calibrando = False
        
        if len(self.dados_calibracao['ear']) > 30:
            self.gerenciador_perfis.atualizar_calibracao(
                self.dados_calibracao['ear'],
                self.dados_calibracao['mar'],
                self.dados_calibracao['blink'],
                self.dados_calibracao['head_stability']
            )
            print("✅ Calibração concluída com sucesso!")
            print(f"   📊 Head Stability médio: {np.mean(self.dados_calibracao['head_stability']):.2f}")
            print(f"   📊 Blink Rate médio: {np.mean(self.dados_calibracao['blink']):.1f}/min")
            print(f"   👁️ EAR médio: {np.mean(self.dados_calibracao['ear']):.3f}")
            print(f"   📊 MAR médio: {np.mean(self.dados_calibracao['mar']):.3f}")
        else:
            print("⚠️ Calibração insuficiente, usando valores padrão")
    
    def desenhar_calibracao(self, frame, tempo_atual):
        """Desenha interface de calibração"""
        tempo_restante = max(0, self.tempo_calibracao_fim - tempo_atual)
        progresso = (30 - tempo_restante) / 30
        
        h, w = frame.shape[:2]
        
# Fundo semi-transparente
        overlay = frame.copy()
        cv2.rectangle(overlay, (w//4, h//4), (3*w//4, 3*h//4), (50, 50, 50), -1)
        frame = cv2.addWeighted(overlay, 0.8, frame, 0.2, 0)
        
# Texto principal
        cv2.putText(frame, "CALIBRACAO AVANCADA", (w//4 + 20, h//4 + 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
        
        cv2.putText(frame, f"Tempo restante: {tempo_restante:.1f}s", (w//4 + 20, h//4 + 100),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        
        cv2.putText(frame, "Mantenha expressao relaxada", (w//4 + 20, h//4 + 130),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        cv2.putText(frame, "Evite movimentos da cabeca", (w//4 + 20, h//4 + 150),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        cv2.putText(frame, "Pisque naturalmente", (w//4 + 20, h//4 + 170),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
# Barra de progresso
        barra_w = w//2
        barra_h = 20
        barra_x = w//4 + 20
        barra_y = h//4 + 200
        
        cv2.rectangle(frame, (barra_x, barra_y), (barra_x + barra_w, barra_y + barra_h), (100, 100, 100), 2)
        cv2.rectangle(frame, (barra_x, barra_y), (barra_x + int(barra_w * progresso), barra_y + barra_h), (0, 255, 0), -1)
        
        return frame
    
    def desenhar_interface_avancada(self, frame, landmarks_px, alerta, metricas, tendencias, thresholds):
        """INTERFACE EXPANDIDA COM TODAS AS MÉTRICAS """
        h, w = frame.shape[:2]
        cor_alerta = alerta['info']['cor']
        texto_alerta = alerta['info']['texto']
        
# Desenhar landmarks com cores baseadas em eventos
        for i in self.OLHO_ESQUERDO + self.OLHO_DIREITO:
            x, y = int(landmarks_px[i][0]), int(landmarks_px[i][1])
# Cor especial se blink detectado
            cor = (0, 255, 255) if metricas.get('blink_detectado', False) else cor_alerta
            cv2.circle(frame, (x, y), 2, cor, -1)
        
        for i in self.BOCA_LANDMARKS:
            x, y = int(landmarks_px[i][0]), int(landmarks_px[i][1])
# Cor especial se bocejo detectado
            cor = (255, 0, 255) if metricas.get('bocejo_detectado', False) else (255, 255, 0)
            cv2.circle(frame, (x, y), 2, cor, -1)
        
# Painel principal expandido (esquerda)
        painel_w = 320  # Aumentado para mais informações
        painel_h = h - 20
        
        overlay = frame.copy()
        cv2.rectangle(overlay, (10, 10), (painel_w, painel_h), (0, 0, 0), -1)
        frame = cv2.addWeighted(overlay, 0.7, frame, 0.3, 0)
        
        y_offset = 40
        
# Status principal
        cv2.putText(frame, f"STATUS: {texto_alerta}", (20, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, cor_alerta, 2)
        y_offset += 40
        
# Score de fadiga
        score = alerta.get('score', 0)
        cv2.putText(frame, f"Fadiga: {score:.0f}/100", (20, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        y_offset += 35
        
# MÉTricas Principais expandidas
        cv2.putText(frame, "METRICAS AVANCADAS:", (20, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
        y_offset += 25
        
# Ear com cor baseada no valor
        ear_cor = (0, 0, 255) if metricas['EAR'] <= 0.2 else (0, 255, 255) if metricas['EAR'] <= 0.25 else (255, 255, 255)
        cv2.putText(frame, f"EAR: {metricas['EAR']:.3f}", (20, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, ear_cor, 1)
        y_offset += 20
        
# Perclos com cor baseada no valor
        perclos_cor = (0, 0, 255) if metricas['PERCLOS'] >= 60 else (0, 255, 255) if metricas['PERCLOS'] >= 30 else (255, 255, 255)
        cv2.putText(frame, f"PERCLOS: {metricas['PERCLOS']:.1f}%", (20, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, perclos_cor, 1)
        y_offset += 20
        
# Mar com detecção de bocejo
        mar_texto = f"MAR: {metricas['MAR']:.3f}"
        if metricas.get('bocejo_detectado', False):
            mar_texto += " (BOCEJO!)"
            mar_cor = (255, 0, 255)
        elif metricas['MAR'] > 0.35:
            mar_cor = (0, 255, 255)
        else:
            mar_cor = (255, 255, 255)
        cv2.putText(frame, mar_texto, (20, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, mar_cor, 1)
        y_offset += 20
        
# Novas MÉTricas
# Blink Rate com avaliação
        br = metricas['BLINK_RATE']
        if br < 5:
            br_texto = f"Blink Rate: {br:.1f}/min (BAIXA)"
            br_cor = (0, 255, 255)
        elif br > 25:
            br_texto = f"Blink Rate: {br:.1f}/min (ALTA)"
            br_cor = (0, 255, 255)
        else:
            br_texto = f"Blink Rate: {br:.1f}/min"
            br_cor = (255, 255, 255)
        cv2.putText(frame, br_texto, (20, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, br_cor, 1)
        y_offset += 20
        
# Head Stability
        hs = metricas['HEAD_STABILITY']
        if hs > 10:
            hs_texto = f"Head Stab: {hs:.1f} (INSTAVEL)"
            hs_cor = (0, 255, 255)
        else:
            hs_texto = f"Head Stab: {hs:.1f}"
            hs_cor = (255, 255, 255)
        cv2.putText(frame, hs_texto, (20, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, hs_cor, 1)
        y_offset += 20
        
        cv2.putText(frame, f"Olhos fechados: {metricas['tempo_olhos_fechados']:.1f}s", (20, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        y_offset += 25
        
# Contadores De Eventos
        cv2.putText(frame, "EVENTOS (sessao):", (20, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
        y_offset += 20
        
        cv2.putText(frame, f"Piscadas: {metricas['total_blinks']}", (20, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        y_offset += 18
        
        cv2.putText(frame, f"Bocejos: {metricas['total_bocejos']}", (20, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        y_offset += 25
        
# Tendências
        if tendencias.get('total_pontos', 0) > 30:
            cv2.putText(frame, "TENDENCIAS (2min):", (20, y_offset),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
            y_offset += 20
            
            tend_perclos = tendencias.get('tendencia_perclos', 'estavel')
            cor_tend = (0, 255, 0) if tend_perclos == 'melhorando' else (0, 0, 255) if tend_perclos == 'deteriorando' else (255, 255, 255)
            
            cv2.putText(frame, f"PERCLOS: {tend_perclos}", (20, y_offset),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, cor_tend, 1)
            y_offset += 18
            
            tend_ear = tendencias.get('tendencia_ear', 'estavel')
            cor_tend = (0, 255, 0) if tend_ear == 'melhorando' else (0, 0, 255) if tend_ear == 'deteriorando' else (255, 255, 255)
            
            cv2.putText(frame, f"EAR: {tend_ear}", (20, y_offset),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, cor_tend, 1)
            y_offset += 18
            
            cv2.putText(frame, f"Taxa alertas: {tendencias.get('taxa_alertas', 0):.1f}%", (20, y_offset),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
            y_offset += 25
        
# Motivos do alerta
        motivos = alerta.get('motivos', [])
        if motivos:
            cv2.putText(frame, "MOTIVOS:", (20, y_offset),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
            y_offset += 20
            
            for motivo in motivos[:4]:  # Máximo 4 motivos
                texto_motivo = motivo[:22] + "..." if len(motivo) > 22 else motivo
                cv2.putText(frame, f"• {texto_motivo}", (20, y_offset),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
                y_offset += 16
        
# Informações de sessão (canto direito)
        estatisticas = self.sistema_alertas.get_estatisticas()
        if estatisticas:
            cv2.putText(frame, f"Alertas: {estatisticas.get('total_alertas', 0)}", (w - 180, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        cv2.putText(frame, f"Frames: {metricas['frames_processados']}", (w - 180, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
# Perfil do usuário
        if self.gerenciador_perfis.perfil_atual:
            nome_usuario = self.gerenciador_perfis.perfil_atual['nome']
            cv2.putText(frame, f"Usuario: {nome_usuario}", (w - 180, 70),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 255, 100), 1)
        
# Indicadores Visuais De Eventos
        if metricas.get('blink_detectado', False):
            cv2.putText(frame, "BLINK!", (w - 180, h - 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        
        if metricas.get('bocejo_detectado', False):
            cv2.putText(frame, "BOCEJO!", (w - 180, h - 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 255), 2)
        
        return frame
    
    def finalizar_sessao(self):
        """Finaliza sessão e salva dados expandidos"""
        relatorio = self.analisador_sessao.gerar_relatorio()
        
# Adicionar estatísticas AvanÇAdas Ao RelatÓRio
        relatorio['total_blinks_sessao'] = len(self.buffer_blink_events)
        relatorio['total_bocejos_sessao'] = len(self.buffer_yawn_events)
        
        if self.gerenciador_perfis.perfil_atual:
            self.gerenciador_perfis.adicionar_sessao(relatorio)
            print("💾 Dados da sessão expandida salvos no perfil")
        
        return relatorio

print("✅ DetectorFadigaInteligente EXPANDIDO criado")

✅ DetectorFadigaInteligente EXPANDIDO criado


In [21]:
# Interface de seleção de usuário
def selecionar_usuario(gerenciador_perfis):
    """Interface para selecionar ou criar usuário"""
    usuarios_existentes = gerenciador_perfis.listar_usuarios()
    
    print("🧠 SISTEMA INTELIGENTE DE DETECÇÃO DE FADIGA")
    print("="*50)
    
    if usuarios_existentes:
        print("👥 Usuários existentes:")
        for i, usuario in enumerate(usuarios_existentes, 1):
            print(f"   {i}. {usuario}")
        
        print(f"   {len(usuarios_existentes) + 1}. Criar novo usuário")
        print("\n💡 Digite o número ou nome do usuário:")
        
        try:
            escolha = input("Escolha: ").strip()
            
# Verificar se é um número
            if escolha.isdigit():
                indice = int(escolha) - 1
                if 0 <= indice < len(usuarios_existentes):
                    nome_usuario = usuarios_existentes[indice]
                    return gerenciador_perfis.carregar_perfil(nome_usuario)
                elif indice == len(usuarios_existentes):
# Criar novo usuário
                    nome_novo = input("Nome do novo usuário: ").strip()
                    if nome_novo:
                        return gerenciador_perfis.criar_perfil(nome_novo)
            else:
# Nome direto
                if escolha in usuarios_existentes:
                    return gerenciador_perfis.carregar_perfil(escolha)
                else:
                    print(f"Criando novo usuário: {escolha}")
                    return gerenciador_perfis.criar_perfil(escolha)
        except:
            pass
    
# Fallback para usuário padrão
    print("Usando usuário padrão: demo")
    return gerenciador_perfis.carregar_perfil("demo")

# Função para detectar câmera
def encontrar_camera_disponivel():
    print("🔍 Procurando câmeras...")
    for i in range(5):
        try:
            cap = cv2.VideoCapture(i)
            if cap.isOpened():
                ret, frame = cap.read()
                if ret and frame is not None and frame.shape[0] > 0:
                    print(f"✅ Câmera encontrada: índice {i}")
                    cap.release()
                    return i
                cap.release()
        except:
            continue
    print("⚠️ Usando câmera padrão (0)")
    return 0

print("✅ Funções auxiliares criadas")

✅ Funções auxiliares criadas


In [ ]:
# Sistema inteligente - execução principal (com câmera)

print("🧠 INICIANDO SISTEMA INTELIGENTE DE DETECÇÃO DE FADIGA")
print("="*60)

# Inicializar gerenciador de perfis
gerenciador_perfis = GerenciadorPerfis()

# Selecionar usuário
print("👤 SELEÇÃO DE USUÁRIO")
perfil = selecionar_usuario(gerenciador_perfis)

# Verificar se precisa calibrar
precisa_calibracao = not perfil['calibracao']['calibrado']
if precisa_calibracao:
    print("\n🎯 PRIMEIRA SESSÃO - CALIBRAÇÃO NECESSÁRIA")
    print("   O sistema irá aprender seus padrões pessoais")
    print("   Duração: 30 segundos")
else:
    print(f"\n✅ USUÁRIO CALIBRADO")
    print(f"   EAR baseline: {perfil['calibracao']['ear_baseline']:.3f}")
    print(f"   Sessões anteriores: {len(perfil['sessoes'])}")

# Configurar câmera - ConfiguraÇÕES Corrigidas
camera_index = encontrar_camera_disponivel()
cap = cv2.VideoCapture(camera_index)

# configurações de display otimizadas
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 800)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 600)
cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 3)  # Buffer maior para estabilidade
cap.set(cv2.CAP_PROP_AUTOFOCUS, 0)   # Desabilitar autofoco se disponível

# Aguardar estabilização da câmera
print("📷 Aguardando estabilização da câmera...")
time.sleep(2)

if not cap.isOpened():
    print("❌ Erro: Não foi possível acessar a câmera")
else:
    print("✅ Câmera configurada e estabilizada")
    
# Teste de leitura para verificar funcionalidade
    ret, test_frame = cap.read()
    if not ret or test_frame is None:
        print("⚠️ Problema na leitura inicial da câmera")
    else:
        print(f"✅ Resolução da câmera: {test_frame.shape[1]}x{test_frame.shape[0]}")
    
# Inicializar detector inteligente
    detector = DetectorFadigaInteligente(gerenciador_perfis)
    
# Iniciar calibração se necessário
    if precisa_calibracao:
        input("\nPressione ENTER quando estiver pronto para calibrar...")
        detector.iniciar_calibracao(30)
    
    print("\n🎮 CONTROLES:")
    print("   'q' - Sair")
    print("   'c' - Recalibrar")
    print("   's' - Estatísticas")
    print("   'r' - Reiniciar sessão")
    print("\n🧠 SISTEMA ATIVO!")
    
    inicio_sessao = time.time()
    
# configuração da janela
    window_name = f'Sistema Inteligente - {perfil["nome"]}'
    cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE)  # WINDOW_AUTOSIZE ao invés de NORMAL
    
    try:
        frame_count = 0
        while True:
            ret, frame = cap.read()
            if not ret or frame is None:
                print("❌ Erro ao capturar frame")
                time.sleep(0.1)  # Pequena pausa antes de tentar novamente
                continue
            
            frame_count += 1
            
# Processar frame
            frame_processado, status, nivel, score, metricas = detector.processar_frame(frame)
            
            
# ExibiÇÃO Otimizada - Sem Redimensionamento ForÇAdo
            cv2.imshow(window_name, frame_processado)
            
# Controles com delay mínimo
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            elif key == ord('c'):  # Recalibrar
                print("🎯 Iniciando recalibração...")
                detector.iniciar_calibracao(20)
            elif key == ord('r'):  # Reiniciar
                detector = DetectorFadigaInteligente(gerenciador_perfis)
                print("🔄 Sistema reiniciado")
            elif key == ord('s'):  # Estatísticas
                stats = detector.sistema_alertas.get_estatisticas()
                tendencias = detector.analisador_sessao.analisar_tendencias()
                
                print(f"\n📊 ESTATÍSTICAS DA SESSÃO:")
                print(f"   ⏱️ Tempo: {time.time() - inicio_sessao:.1f}s")
                print(f"   🎬 Frames: {metricas.get('frames_processados', 0)}")
                print(f"   🚨 Alertas: {stats.get('total_alertas', 0)}")
                print(f"   📈 EAR atual: {metricas.get('EAR', 0):.3f}")
                print(f"   📊 PERCLOS atual: {metricas.get('PERCLOS', 0):.1f}%")
                print(f"   🎯 Status: {status}")
                if tendencias.get('total_pontos', 0) > 30:
                    print(f"   📈 Tendência PERCLOS: {tendencias.get('tendencia_perclos', 'N/A')}")
                    print(f"   👁️ Tendência EAR: {tendencias.get('tendencia_ear', 'N/A')}")
    
    except KeyboardInterrupt:
        print("\n🛑 Interrompido pelo usuário")
    
    finally:
# Finalizar sessão
        relatorio = detector.finalizar_sessao()
        
        print(f"\n🧠 SESSÃO INTELIGENTE FINALIZADA")
        print(f"   👤 Usuário: {perfil['nome']}")
        print(f"   ⏱️ Duração: {relatorio.get('duracao', 0):.1f}s")
        print(f"   🚨 Alertas: {relatorio.get('total_alertas', 0)}")
        print(f"   🎬 Frames processados: {frame_count}")
        
        if 'EAR_medio' in relatorio:
            print(f"   👁️ EAR médio: {relatorio['EAR_medio']:.3f}")
        if 'PERCLOS_medio' in relatorio:
            print(f"   📊 PERCLOS médio: {relatorio['PERCLOS_medio']:.1f}%")
        
# estatísticas do perfil
        total_sessoes = perfil['estatisticas']['total_sessoes']
        tempo_total = perfil['estatisticas']['tempo_total_uso']
        print(f"\n📈 ESTATÍSTICAS DO PERFIL:")
        print(f"   📊 Total de sessões: {total_sessoes + 1}")
        print(f"   ⏱️ Tempo total de uso: {(tempo_total + relatorio.get('duracao', 0))/60:.1f} min")
        
# Limpeza Adequada
        cap.release()
        cv2.destroyAllWindows()
        cv2.waitKey(1)  # Força limpeza das janelas
        print("✅ Sistema finalizado - câmera liberada")

🧠 INICIANDO SISTEMA INTELIGENTE DE DETECÇÃO DE FADIGA
👤 SELEÇÃO DE USUÁRIO
🧠 SISTEMA INTELIGENTE DE DETECÇÃO DE FADIGA
👥 Usuários existentes:
   1. Edu
   2. Usuario_Padrao
   3. demo
   4. edu
   5. edu1
   6. teste00
   7. usuario1
   8. Criar novo usuário

💡 Digite o número ou nome do usuário:


Escolha:  Edu


Usando usuário padrão: demo
✅ Perfil carregado para demo
   📊 6 sessões anteriores
   🎯 Calibrado: Sim

✅ USUÁRIO CALIBRADO
   EAR baseline: 0.271
   Sessões anteriores: 6
🔍 Procurando câmeras...
✅ Câmera encontrada: índice 0
📷 Aguardando estabilização da câmera...
✅ Câmera configurada e estabilizada
✅ Resolução da câmera: 800x600
🔊 Sistema de alerta sonoro inicializado com pygame
⚠️ Erro ao criar sons: operands could not be broadcast together with shap
🤖 Modelo ML carregado
🧠 Detector Inteligente Inicializado - Regras Avançadas

🎮 CONTROLES:
   'q' - Sair
   'c' - Recalibrar
   's' - Estatísticas
   'r' - Reiniciar sessão

🧠 SISTEMA ATIVO!
🔄 Mudança de nível para: NORMAL (consenso: 5/6)
🔄 Mudança de nível para: NORMAL (consenso: 6/6)
🔄 Mudança de nível para: NORMAL (consenso: 6/6)
🔄 Mudança de nível para: NORMAL (consenso: 6/6)
🔄 Mudança de nível para: NORMAL (consenso: 5/6)
🔊 Beep Linux: AVISO (800Hz)
🔄 Mudança de nível para: AVISO (consenso: 5/6)
🔄 Mudança de nível para: AVISO (cons

# 📋 GUIA DO SISTEMA INTELIGENTE

## Características Avançadas

### Sistema de Perfis Personalizados
- **Calibração Automática**: 30s para aprender padrões pessoais
- **Thresholds Adaptativos**: Ajustados aos dados de cada usuário
- **Histórico Persistente**: Dados salvos entre sessões
- **Múltiplos Usuários**: Cada pessoa tem seu perfil único

### Sistema de Alertas Progressivos
1. **NORMAL** 🟢 - Funcionamento normal
2. **AVISO** 🟡 - Primeiros sinais (PERCLOS >30%)
3. **ATENÇÃO** 🟠 - Sinais moderados (PERCLOS >45%)
4. **ALERTA** 🔴 - Sinais claros (PERCLOS >60%)
5. **CRÍTICO** 🔴 - Situação perigosa (PERCLOS >80%)

### Análise de Tendências
- **Tendências Temporais**: Analisa últimos 2 minutos
- **Padrões Comportamentais**: Detecta deterioração gradual
- **Predição Precoce**: Alertas antes da fadiga crítica

### Regras Complexas Adaptativas
- **Score de Fadiga**: Combinação ponderada de múltiplas métricas
- **Contexto Temporal**: Considera histórico recente
- **Personalização**: Thresholds únicos por usuário
- **Aprendizado**: Melhora com mais dados

## 🎮 Como Usar

### 1️⃣ Primeira Vez
1. Execute o sistema
2. Digite seu nome quando solicitado
3. Aguarde calibração automática (30s)
4. Mantenha expressão relaxada durante calibração

### 2️⃣ Sessões Subsequentes
1. Sistema carrega seu perfil automaticamente
2. Thresholds já personalizados
3. Histórico considerado na detecção

### 3️⃣ Controles Avançados
- **'c'**: Recalibrar perfil
- **'s'**: Ver estatísticas detalhadas
- **'r'**: Reiniciar sessão
- **'q'**: Sair (salva dados automaticamente)

## Interface Inteligente

### Painel Esquerdo
- **Status atual** com cor codificada
- **Score de fadiga** (0-100)
- **Métricas em tempo real**
- **Análise de tendências**
- **Motivos dos alertas**
- **Thresholds personalizados**

### Painel Direito
- **Estatísticas da sessão**
- **Nome do usuário**
- **Contador de frames**

## Para Demonstrações

### Testar Personalização
1. Crie perfil "Visitante1", "Visitante2", etc.
2. Cada um terá calibração única
3. Compare thresholds entre pessoas
4. Mostre histórico de sessões

### Demonstrar Inteligência
- **Tendências**: Feche olhos gradualmente
- **Score Progressivo**: Varie intensidade
- **Alertas Personalizados**: Cada pessoa tem thresholds diferentes
- **Histórico**: Sessões anteriores influenciam detecção

## Dados Salvos

### Perfis (`perfis_usuarios/`)
- **Nome.json**: Dados completos do usuário
- **Calibração**: Thresholds personalizados
- **Histórico**: Últimas 50 sessões
- **Estatísticas**: Uso acumulado

### Vantagens
- **Melhora com uso**: Mais preciso a cada sessão
- **Adaptação individual**: Cada pessoa é única
- **Análise longitudinal**: Detecta mudanças ao longo do tempo
- **Sistema completo**: Pronto para uso profissional